# Qwen2.5 Opposing Counsel Training Notebook

Upgraded training pipeline with:
- Assistant-only loss masking
- Packing enabled
- Better LoRA target modules
- EOS handling
- Improved SFTConfig setup
- Resume-safe training

In [ ]:

!pip install -q transformers datasets peft trl bitsandbytes accelerate sentencepiece


In [ ]:

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

print("Torch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

# =========================
# CONFIG
# =========================

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

DATASET_PATH = "dataset.jsonl"

OUTPUT_DIR = "./qwen-opposing-counsel"

MAX_SEQ_LENGTH = 1024

BATCH_SIZE = 1
GRAD_ACCUM = 8

LEARNING_RATE = 2e-4
NUM_EPOCHS = 3

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

print("Config loaded")


In [ ]:

# =========================
# LOAD DATASET
# =========================

dataset = load_dataset(
    "json",
    data_files=DATASET_PATH,
    split="train"
)

print(dataset)

# Train / test split
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(dataset)


In [ ]:

# =========================
# LOAD TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded")


In [ ]:

# =========================
# FORMAT FUNCTION
# =========================

def formatting_prompts_func(example):

    messages = example["messages"]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    # Important
    # Explicit EOS improves stopping behavior
    return text + tokenizer.eos_token


In [ ]:

# =========================
# 4-BIT QUANTIZATION
# =========================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

print("Model loaded")


In [ ]:

# =========================
# LoRA CONFIG
# =========================

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()


In [ ]:

# =========================
# TRAINER CONFIG
# =========================

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,

    logging_steps=10,

    save_steps=100,
    save_total_limit=2,

    evaluation_strategy="steps",
    eval_steps=100,

    bf16=USE_BF16,
    fp16=not USE_BF16,

    gradient_checkpointing=True,

    max_seq_length=MAX_SEQ_LENGTH,

    packing=True,

    # MOST IMPORTANT CHANGE
    assistant_only_loss=True,

    optim="paged_adamw_8bit",

    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    report_to="none",
)

print("SFT config ready")


In [ ]:

# =========================
# TRAINER
# =========================

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,
    args=sft_config,
)

print("Trainer ready")


In [ ]:

# =========================
# AUTO RESUME TRAINING
# =========================

from transformers.trainer_utils import get_last_checkpoint

checkpoint = None

if os.path.isdir(OUTPUT_DIR):
    checkpoint = get_last_checkpoint(OUTPUT_DIR)

if checkpoint:
    print(f"Resuming from checkpoint: {checkpoint}")
    trainer.train(resume_from_checkpoint=checkpoint)
else:
    print("Starting fresh training")
    trainer.train()


In [ ]:

# =========================
# SAVE MODEL
# =========================

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved")


In [ ]:

# =========================
# INFERENCE TEST
# =========================

from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

messages = [
    {
        "role": "system",
        "content": "You are an aggressive opposing legal counsel."
    },
    {
        "role": "user",
        "content": "My client should not be liable because there was no written agreement."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

output = pipe(
    prompt,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
)

print(output[0]["generated_text"])
